In [ ]:
!pip install roboflow ultralytics albumentations -q

from roboflow import Roboflow
rf = Roboflow(api_key="api_key")
project = rf.workspace("js-ozptv").project("foggy-car-ofcf4")
version = project.version(1)
dataset = version.download("yolo26")  # yolov8 format works with ultralytics

DATASET_PATH = dataset.location
print(f"Dataset at: {DATASET_PATH}")

In [ ]:
import os
from collections import defaultdict
import numpy as np

def dataset_sanity_check(dataset_path, split="train"):
    images_dir = os.path.join(dataset_path, split, "images")
    labels_dir = os.path.join(dataset_path, split, "labels")

    image_files = set(os.listdir(images_dir))
    label_files = set(os.listdir(labels_dir))

    image_basenames = {os.path.splitext(f)[0] for f in image_files}
    label_basenames = {os.path.splitext(f)[0] for f in label_files}

    missing_labels = image_basenames - label_basenames
    orphan_labels = label_basenames - image_basenames
    empty_labels = []
    total_boxes = 0

    for lbl in label_files:
        with open(os.path.join(labels_dir, lbl)) as f:
            lines = f.readlines()
            if len(lines) == 0:
                empty_labels.append(lbl)
            total_boxes += len(lines)

    print(f"\n Split: {split.upper()}")
    print(f"   Images: {len(image_files)}")
    print(f"   Label files: {len(label_files)}")
    print(f"   Total bounding boxes: {total_boxes}")
    print(f"   Images without labels: {len(missing_labels)}")
    print(f"   Orphan label files: {len(orphan_labels)}")
    print(f"   Empty label files: {len(empty_labels)}")
    return {"missing_labels": missing_labels, "orphan_labels": orphan_labels, "empty_labels": empty_labels}

for split in ["train", "valid", "test"]:
    dataset_sanity_check(DATASET_PATH, split)

In [ ]:
import matplotlib.pyplot as plt

def class_distribution(dataset_path, split="train"):
    labels_dir = os.path.join(dataset_path, split, "labels")
    class_counts = defaultdict(int)
    for lbl in os.listdir(labels_dir):
        with open(os.path.join(labels_dir, lbl)) as f:
            for line in f:
                class_id = int(line.split()[0])
                class_counts[class_id] += 1
    return class_counts

import yaml
with open(os.path.join(DATASET_PATH, "data.yaml")) as f:
    data = yaml.safe_load(f)
class_names = data["names"]
print(f"Classes: {class_names}")

train_counts = class_distribution(DATASET_PATH, "train")

fig, ax = plt.subplots(figsize=(10, 4))
classes = [class_names[k] for k in sorted(train_counts.keys())]
counts = [train_counts[k] for k in sorted(train_counts.keys())]
bars = ax.bar(classes, counts, color='steelblue', edgecolor='black')
ax.bar_label(bars)
ax.set_title("Train Class Distribution", fontsize=14)
ax.set_xlabel("Class")
ax.set_ylabel("Number of Instances")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Imbalance ratio
max_count = max(counts)
print("\n Class Imbalance Ratios:")
for name, count in zip(classes, counts):
    print(f"   {name}: {count} samples | ratio: 1:{max_count//count}")

In [ ]:
import shutil

train_images = os.path.join(DATASET_PATH, "train/images")
train_labels = os.path.join(DATASET_PATH, "train/labels")

# Dynamically find minority classes (bottom 40% by count)
sorted_counts = sorted(train_counts.items(), key=lambda x: x[1])
threshold = np.percentile(list(train_counts.values()), 40)
minority_classes = [cls_id for cls_id, cnt in train_counts.items() if cnt <= threshold]

print(f"Minority classes to oversample: {[class_names[c] for c in minority_classes]}")

duplicated = 0
for label_file in os.listdir(train_labels):
    label_path = os.path.join(train_labels, label_file)
    with open(label_path, "r") as f:
        lines = f.readlines()

    contains_minority = any(int(line.split()[0]) in minority_classes for line in lines)

    if contains_minority:
        image_file = label_file.replace(".txt", ".jpg")
        for i in range(2):  # duplicate 2x for stronger minority boost
            new_image_name = image_file.replace(".jpg", f"_dup{i}.jpg")
            new_label_name = label_file.replace(".txt", f"_dup{i}.txt")
            src_img = os.path.join(train_images, image_file)
            if os.path.exists(src_img):
                shutil.copy(src_img, os.path.join(train_images, new_image_name))
                shutil.copy(label_path, os.path.join(train_labels, new_label_name))
                duplicated += 1

print(f"Duplicated {duplicated} minority class samples")

In [ ]:
import cv2
import albumentations as A
import numpy as np

def get_fog_border_color(img: np.ndarray) -> tuple:
    """
    Sample the average color of the top 10% of the image (sky/fog region).
    Used for CoarseDropout fill so patches blend with the foggy sky.
    """
    h = img.shape[0]
    sky_region = img[:max(1,h//10),:,:]
    mean_color = sky_region.mean(axis=(0,1))
    return tuple(int(c) for c in mean_color)


def get_augmentation_pipeline(fog_color: tuple=(180,180,180)):
    """Augmentation pipeline for the dataset"""

    r,g,b = fog_color

    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomResizedCrop(
            size=(640, 640),
            scale=(0.80, 1.0),
            ratio=(1.0, 1.0),    # preserve aspect ratio
            p=0.4
        ),
        A.RandomBrightnessContrast(brightness_limit=0.15,
                                   contrast_limit=0.15,
                                   p=0.5),
        A.RandomGamma(gamma_limit=(90,110),p=0.3),
        # random_gamma randomly changes the brightness/contrast of an img using gamma correction
        # gamma_limit=(90,110) picks random gamma values val < 1.0 -> brighter img, val > 1.0 -> darker img
        A.CLAHE(clip_limit=2.0,
                tile_grid_size=(8,8),
                p=0.3),
        # CLAHE stands for Contrast Limit Adaptive Histogram Equivalent
        # It enhances contrast locally instead of globally
        # clip_limit=2.0 limits contrast prevents noise from being over enhanced
        # tile_grid_size=(8,8) splits img into (8,8) small blocks applies histogram equivalization to each block
        # smaller tiles: stronger local contrast,larger_tiles: smoother effect
        A.HueSaturationValue(
            hue_shift_limit=5,
            sat_shift_limit=15,
            val_shift_limit=10,
            p=0.3
        ),
        # HueSaturationValue randomly changes the color properties of img
        # hue_shift_limit changes the color tone
        # sat_shift_limit changes color intensity
        # val_shift_limit changes the brightness
        A.GaussianBlur(blur_limit=(3,3),p=0.3),
        # adds blur using gaussian filter using (3,3) filter
        # small blur realistic camera blur
        A.CoarseDropout(num_holes_range=(1, 3),
                        hole_height_range=(10, 20),
                        hole_width_range=(10, 20),
                        fill=(r, g, b)) # fog sky color as fill p=0.2
        # CoarseDropout randomly hides small rectangular patches of an img
    ],bbox_params=A.BboxParams(
        format="yolo",
        label_fields=['class_labels'],
        min_visibility=0.3
    ))

def apply_augmentations_to_dataset(
    images_dir, labels_dir,
    output_images_dir, output_labels_dir,
    n_augments=2
):
    """Generate n_augments augmented copies of every image+label pair."""
    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)

    augmented = 0
    skipped = 0

    for img_file in os.listdir(images_dir):
        if not img_file.endswith(('.jpg', '.png', '.jpeg')):
            continue

        img_path = os.path.join(images_dir, img_file)
        lbl_path = os.path.join(labels_dir, os.path.splitext(img_file)[0] + '.txt')

        if not os.path.exists(lbl_path):
            skipped += 1
            continue

        img = cv2.imread(img_path)
        if img is None:
            skipped += 1
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        fog_color = get_fog_border_color(img)
        pipeline = get_augmentation_pipeline(fog_color=fog_color)

        with open(lbl_path) as f:
            lines = f.readlines()

        bboxes, class_labels = [], []
        for line in lines:
            parts = list(map(float, line.strip().split()))
            class_labels.append(int(parts[0]))
            bboxes.append(parts[1:5])

        for aug_idx in range(n_augments):
            try:
                result = pipeline(image=img, bboxes=bboxes, class_labels=class_labels)
                if len(result['bboxes']) == 0:
                    continue

                aug_img = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
                base_name = os.path.splitext(img_file)[0]

                cv2.imwrite(
                    os.path.join(output_images_dir, f"{base_name}_aug{aug_idx}.jpg"),
                    aug_img,
                    [cv2.IMWRITE_JPEG_QUALITY, 95]
                )
                with open(os.path.join(output_labels_dir, f"{base_name}_aug{aug_idx}.txt"), 'w') as f:
                    for cls, bbox in zip(result['class_labels'], result['bboxes']):
                        f.write(f"{cls} {' '.join(f'{v:.6f}' for v in bbox)}\n")

                augmented += 1
            except Exception as e:
                skipped += 1

    print(f"Done. Generated: {augmented} | Skipped: {skipped}")


def preview_augmentations(images_dir, labels_dir, n=3):
    """
    Preview augmented vs original side by side.
    What to verify:
      - No black corners, no grey strips on any edge
      - CoarseDropout patches match the foggy sky, not solid black
      - Bounding box count preserved (min_visibility=0.3 filter)
    """
    files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')][:n]

    fig, axes = plt.subplots(n, 2, figsize=(12, 4 * n))
    if n == 1:
        axes = [axes]

    for i, img_file in enumerate(files):
        img = cv2.imread(os.path.join(images_dir, img_file))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        fog_color = get_fog_border_color(img)
        pipeline = get_augmentation_pipeline(fog_color=fog_color)

        lbl_path = os.path.join(labels_dir, os.path.splitext(img_file)[0] + '.txt')
        bboxes, class_labels = [], []
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f.readlines():
                    parts = list(map(float, line.strip().split()))
                    class_labels.append(int(parts[0]))
                    bboxes.append(parts[1:5])

        result = pipeline(image=img, bboxes=bboxes, class_labels=class_labels)
        aug_img = result['image']

        axes[i][0].imshow(img)
        axes[i][0].set_title(f"Original | fog std: {img.std():.1f} | fill: {fog_color}")
        axes[i][0].axis('off')

        axes[i][1].imshow(aug_img)
        axes[i][1].set_title(f"Augmented | boxes: {len(result['bboxes'])}/{len(bboxes)}")
        axes[i][1].axis('off')

    plt.suptitle("Augmentation preview — no border artifacts, fog-colored dropout", fontsize=11)
    plt.tight_layout()
    plt.show()


preview_augmentations(
    os.path.join(DATASET_PATH, "train/images"),
    os.path.join(DATASET_PATH, "train/labels")
)

In [ ]:
import torch
import numpy as np
from ultralytics import YOLO
from ultralytics.data.dataset import YOLODataset
import ultralytics.data.build as build


class YOLOWeightedDataset(YOLODataset):
    """
    Improved weighted sampler:
    - Uses MAX aggregation (not mean) → rare class images get priority
    - Temperature scaling on weights → controls sampling sharpness
    """
    def __init__(self, *args, mode="train", temperature=1.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_mode = "train" in self.prefix
        self.temperature = temperature  # > 1 = smoother, < 1 = sharper

        self.count_instances()
        class_weights = np.sum(self.counts) / (self.counts + 1e-6)

        # Temperature scaling for smoother distribution
        class_weights = class_weights ** (1.0 / self.temperature)
        self.class_weights = class_weights
        self.weights = self.calculate_weights()
        self.probabilities = self.calculate_probabilities()

        print(f"[WeightedDataset] Class counts: {self.counts}")
        print(f"[WeightedDataset] Class weights: {self.class_weights.round(3)}")

    def count_instances(self):
        self.counts = np.zeros(len(self.data["names"]), dtype=np.float32)
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)
            for id in cls:
                self.counts[id] += 1
        self.counts = np.where(self.counts == 0, 1, self.counts)

    def calculate_weights(self):
        weights = []
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)
            if cls.size == 0:
                weights.append(1.0)
                continue
            # MAX aggregation: image weight = rarest class in it
            weight = np.max(self.class_weights[cls])
            weights.append(float(weight))
        return weights

    def calculate_probabilities(self):
        total = sum(self.weights)
        return [w / total for w in self.weights]

    def __getitem__(self, index):
        if not self.train_mode:
            return self.transforms(self.get_image_and_label(index))
        # Weighted sampling during training
        index = np.random.choice(len(self.labels), p=self.probabilities)
        return self.transforms(self.get_image_and_label(index))


# Inject improved dataset
build.YOLODataset = YOLOWeightedDataset
print(" Injected YOLOWeightedDataset")

In [ ]:
# ✅ PyTorch performance flags
torch.backends.cuda.matmul.fp32_precision = "tf32"
torch.backends.cudnn.conv.fp32_precision = "tf32"
torch.backends.cudnn.benchmark = True

model = YOLO("yolo26m.pt")

results = model.train(
    data=os.path.join(DATASET_PATH, "data.yaml"),
    epochs=150,
    imgsz=640,
    batch=64,

    # ✅ Use MuSGD — YOLO26's native optimizer
    # Simply don't pass optimizer= at all, OR explicitly pass:
    optimizer="MuSGD",
    lr0=0.01,          # MuSGD works best with SGD-range LR (0.01), NOT AdamW-range (0.001)
    momentum=0.937,    # standard SGD momentum
    weight_decay=5e-4,
    cos_lr=True,

    warmup_epochs=5.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.05,

    # Loss weights
    box=7.5,
    cls=0.5,
    # Note: dfl= has no effect in YOLO26 — DFL was removed from the architecture

    # Augmentations
    mosaic=1.0,
    close_mosaic=15,
    mixup=0.1,
    copy_paste=0.2,
    hsv_s=0.5,
    hsv_v=0.5,
    scale=0.5,
    translate=0.1,
    fliplr=0.5,
    degrees=5.0,
    erasing=0.2,

    # Performance
    cache="disk",
    amp=True,
    pretrained=True,
    freeze=0,
    workers=8,
    patience=40,
    plots=True,
    project="runs/detect",
    name="fog_vehicle_musgd",
)

In [ ]:
# Standard evaluation
metrics_standard = model.val(
    data=os.path.join(DATASET_PATH, "data.yaml"),
    split="test",
    imgsz=640,
    conf=0.001,   # Low conf for val → NMS handles filtering
    iou=0.6,
)

# TTA evaluation
metrics_tta = model.val(
    data=os.path.join(DATASET_PATH, "data.yaml"),
    split="test",
    imgsz=640,
    augment=True,  # ← enables TTA
    conf=0.001,
    iou=0.6,
)

print("\n Results Comparison")
print(f"{'Metric':<20} {'Standard':>12} {'With TTA':>12}")
print("-" * 46)
print(f"{'mAP50':<20} {metrics_standard.box.map50:>12.4f} {metrics_tta.box.map50:>12.4f}")
print(f"{'mAP50-95':<20} {metrics_standard.box.map:>12.4f} {metrics_tta.box.map:>12.4f}")
print(f"{'Precision':<20} {metrics_standard.box.p.mean():>12.4f} {metrics_tta.box.p.mean():>12.4f}")
print(f"{'Recall':<20} {metrics_standard.box.r.mean():>12.4f} {metrics_tta.box.r.mean():>12.4f}")

In [ ]:
# Sweep IoU thresholds to find best NMS setting
iou_thresholds = [0.45, 0.5, 0.55, 0.6, 0.65, 0.7]
results_sweep = []

for iou_thresh in iou_thresholds:
    m = model.val(
        data=os.path.join(DATASET_PATH, "data.yaml"),
        split="val",
        imgsz=640,
        conf=0.001,
        iou=iou_thresh,
        verbose=False,
    )
    results_sweep.append({
        "iou": iou_thresh,
        "mAP50": m.box.map50,
        "mAP50-95": m.box.map,
        "precision": m.box.p.mean(),
        "recall": m.box.r.mean(),
    })
    print(f"IoU={iou_thresh:.2f} → mAP50={m.box.map50:.4f} | mAP50-95={m.box.map:.4f} | P={m.box.p.mean():.4f} | R={m.box.r.mean():.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ious = [r["iou"] for r in results_sweep]
axes[0].plot(ious, [r["mAP50"] for r in results_sweep], 'b-o', label='mAP50')
axes[0].plot(ious, [r["mAP50-95"] for r in results_sweep], 'r-o', label='mAP50-95')
axes[0].set_xlabel("NMS IoU Threshold")
axes[0].set_ylabel("mAP")
axes[0].set_title("mAP vs NMS IoU")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(ious, [r["precision"] for r in results_sweep], 'g-o', label='Precision')
axes[1].plot(ious, [r["recall"] for r in results_sweep], 'm-o', label='Recall')
axes[1].set_xlabel("NMS IoU Threshold")
axes[1].set_ylabel("Score")
axes[1].set_title("Precision/Recall vs NMS IoU")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

best = max(results_sweep, key=lambda x: x["mAP50-95"])
print(f"\n Best NMS IoU: {best['iou']} → mAP50-95: {best['mAP50-95']:.4f}")

In [ ]:
conf_thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
conf_results = []

for conf in conf_thresholds:
    m = model.val(
        data=os.path.join(DATASET_PATH, "data.yaml"),
        split="val",
        imgsz=640,
        conf=conf,
        iou=best["iou"],  # use best NMS IoU from above
        verbose=False,
    )
    conf_results.append({
        "conf": conf,
        "precision": m.box.p.mean(),
        "recall": m.box.r.mean(),
        "mAP50-95": m.box.map,
    })

# F1 = 2 * P * R / (P + R)
for r in conf_results:
    p, rec = r["precision"], r["recall"]
    r["f1"] = 2 * p * rec / (p + rec + 1e-6)
    print(f"conf={r['conf']:.2f} → P={p:.3f} | R={rec:.3f} | F1={r['f1']:.3f} | mAP50-95={r['mAP50-95']:.4f}")

best_conf = max(conf_results, key=lambda x: x["f1"])
print(f"\nBest confidence threshold: {best_conf['conf']} (F1={best_conf['f1']:.4f})")

# PR curve
plt.figure(figsize=(8, 5))
plt.plot([r["recall"] for r in conf_results], [r["precision"] for r in conf_results], 'b-o')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Tradeoff (varying confidence)")
plt.grid(True)
plt.show()

In [ ]:
final_metrics = model.val(
    data=os.path.join(DATASET_PATH, "data.yaml"),
    split="test",
    imgsz=640,
    conf=best_conf["conf"],
    iou=best["iou"],
    augment=True,  # TTA
)

# Per-class breakdown
print("\n Per-Class Performance:")
print(f"{'Class':<20} {'AP50':>8} {'AP50-95':>10} {'Precision':>10} {'Recall':>8}")
print("-" * 60)
for i, name in enumerate(class_names):
    try:
        ap50 = final_metrics.box.ap50[i]
        ap = final_metrics.box.ap[i]
        p = final_metrics.box.p[i]
        r = final_metrics.box.r[i]
        print(f"{name:<20} {ap50:>8.4f} {ap:>10.4f} {p:>10.4f} {r:>8.4f}")
    except:
        pass

print(f"\n{'OVERALL':<20} {final_metrics.box.map50:>8.4f} {final_metrics.box.map:>10.4f}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
try:
    axes[0].bar(class_names, final_metrics.box.ap50, color='steelblue')
    axes[0].set_title("Per-Class AP50")
    axes[0].set_ylabel("AP50")
    axes[0].tick_params(axis='x', rotation=30)

    axes[1].bar(class_names, final_metrics.box.ap, color='coral')
    axes[1].set_title("Per-Class AP50-95")
    axes[1].set_ylabel("AP50-95")
    axes[1].tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()
except:
    print("(Could not plot per-class — check metrics object structure)")

In [ ]:
import cv2
import numpy as np

def estimate_fog_level(img_path):
    """Estimate fog density using contrast (std of grayscale)."""
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    return img.std() if img is not None else 999

def predict_on_image(model, img_path, conf=0.2, iou=0.5):
    results = model.predict(source=img_path, conf=conf, iou=iou, verbose=False)
    return results[0]

def draw_detections(img_path, result, class_names):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        label = f"{class_names[cls]} {conf:.2f}"
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, label, (x1, max(y1-5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return img

# Sample images and split by fog level
test_images_dir = os.path.join(DATASET_PATH, "test/images")
all_test_imgs = [f for f in os.listdir(test_images_dir) if f.endswith('.jpg')]

fog_scores = [(f, estimate_fog_level(os.path.join(test_images_dir, f))) for f in all_test_imgs]
fog_scores.sort(key=lambda x: x[1])

foggy_samples = fog_scores[:3]   # lowest contrast = heaviest fog
clear_samples = fog_scores[-3:]  # highest contrast = clearest

print("Foggy images (std):", [(f, round(s, 2)) for f, s in foggy_samples])
print("Clear images (std):", [(f, round(s, 2)) for f, s in clear_samples])

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for col, (fname, score) in enumerate(foggy_samples):
    img_path = os.path.join(test_images_dir, fname)
    result = predict_on_image(model, img_path)
    drawn = draw_detections(img_path, result, class_names)
    axes[0][col].imshow(drawn)
    axes[0][col].set_title(f" Foggy | std={score:.1f}")
    axes[0][col].axis('off')

for col, (fname, score) in enumerate(clear_samples):
    img_path = os.path.join(test_images_dir, fname)
    result = predict_on_image(model, img_path)
    drawn = draw_detections(img_path, result, class_names)
    axes[1][col].imshow(drawn)
    axes[1][col].set_title(f" Clear | std={score:.1f}")
    axes[1][col].axis('off')

plt.suptitle("Fog vs Clear Detection Comparison", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from PIL import Image

# Training curves
results_csv = pd.read_csv("/content/runs/detect/runs/detect/fog_vehicle_musgd4/results.csv")
results_csv.columns = results_csv.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0][0].plot(results_csv['metrics/mAP50(B)'], label='mAP50', color='blue')
axes[0][0].plot(results_csv['metrics/mAP50-95(B)'], label='mAP50-95', color='red')
axes[0][0].set_title('mAP over Epochs')
axes[0][0].legend(); axes[0][0].grid(True)

axes[0][1].plot(results_csv['metrics/precision(B)'], label='Precision', color='green')
axes[0][1].plot(results_csv['metrics/recall(B)'], label='Recall', color='orange')
axes[0][1].set_title('Precision & Recall')
axes[0][1].legend(); axes[0][1].grid(True)

axes[0][2].plot(results_csv['train/box_loss'], label='Train Box Loss')
axes[0][2].plot(results_csv['val/box_loss'], label='Val Box Loss')
axes[0][2].set_title('Box Loss')
axes[0][2].legend(); axes[0][2].grid(True)

axes[1][0].plot(results_csv['train/cls_loss'], label='Train Cls Loss')
axes[1][0].plot(results_csv['val/cls_loss'], label='Val Cls Loss')
axes[1][0].set_title('Classification Loss')
axes[1][0].legend(); axes[1][0].grid(True)

axes[1][1].plot(results_csv['train/dfl_loss'], label='Train DFL Loss')
axes[1][1].plot(results_csv['val/dfl_loss'], label='Val DFL Loss')
axes[1][1].set_title('DFL Loss (Box Regression Quality)')
axes[1][1].legend(); axes[1][1].grid(True)

axes[1][2].plot(results_csv['lr/pg0'], label='LR group 0')
axes[1][2].set_title('Learning Rate Schedule')
axes[1][2].legend(); axes[1][2].grid(True)

plt.tight_layout()
plt.savefig('training_summary.png', dpi=150)
plt.show()
print("Saved training_summary.png")

In [ ]:
! ffmpeg -i /content/46_hazy_video.mp4 \
-vcodec libx264 -crf 28 -preset slow \
-acodec aac -b:a 128k \
/content/46_hazy_video_compressed.mp4


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/runs/detect/fog_vehicle_musgd4/weights/best.pt")
results = model.predict(
    source="/content/46_hazy_video_compressed.mp4",
    imgsz=640,
    conf=0.1,
    save=True,
    device=0
)


In [ ]:
!ffmpeg -i /content/runs/detect/predict/46_hazy_video_compressed.avi \
-vcodec libx264 -crf 23 -preset fast \
/content/output_fixed.mp4

In [ ]:
!zip -r runs.zip /content/runs

In [ ]:
from google.colab import files
files.download('runs.zip')

In [ ]:
from IPython.display import Video
Video("/content/output_fixed.mp4", embed=True)
